# Logistic regression cho churn: đọc hệ số và lift cho Head of Retention

**Bối cảnh.** Head of Retention đang chọn khách để gọi giữ chân bằng một quy tắc (month-to-month, gắn bó tối đa 12
tháng) và hỏi hai câu: *"Những yếu tố nào gắn với việc khách rời bỏ, mạnh cỡ nào, nói bằng ngôn ngữ của tôi?"* và
*"Nếu dùng model thay cho quy tắc, đội gọi của tôi được lợi gì?"*

Dữ liệu: bộ `telco-churn` (IBM, 7 043 khách, 26.5% rời bỏ). Bạn dựng 10 đặc trưng có nghĩa kinh doanh, fit logistic
regression bằng `statsmodels` (để có khoảng tin cậy và p-value), dịch hệ số thành odds ratio, rồi đo lift theo decile
trên tập test và so với quy tắc.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm hoặc hằng. Điền vào chỗ `# TODO`; hàm không được sửa DataFrame đầu vào.
- Cell gắn tag `explore` đọc dữ liệu thật, fit, in kết quả; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/14-ml-bigdata-for-analysts/exercises/intermediate/01-churn-logistic -q`. Xong thì làm phần **trắc nghiệm** trong app.

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

raw = pd.read_csv(ensure("telco-churn") / "WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(raw.shape)
for col in ["Contract", "InternetService", "PaymentMethod", "TechSupport", "PaperlessBilling"]:
    print(raw[col].value_counts().to_dict())

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import train_test_split

## 1. Đặc trưng có nghĩa kinh doanh

Logistic regression đọc được khi mỗi cột có một **nhóm tham chiếu** rõ và biến số có **đơn vị** người nghe hiểu.
`build_features(raw) -> (X, y)`:

| Cột của `X` | Cách tính | Nhóm tham chiếu / đơn vị |
| --- | --- | --- |
| `contract_one_year` | `Contract == "One year"` | so với month-to-month |
| `contract_two_year` | `Contract == "Two year"` | so với month-to-month |
| `tenure_years` | `tenure / 12` | mỗi năm gắn bó |
| `fiber_optic` | `InternetService == "Fiber optic"` | so với DSL |
| `no_internet` | `InternetService == "No"` | so với DSL |
| `electronic_check` | `PaymentMethod == "Electronic check"` | so với ba cách trả khác |
| `monthly_charges_10` | `MonthlyCharges / 10` | mỗi 10 USD cước tháng |
| `senior` | `SeniorCitizen` (0/1) | |
| `tech_support` | `TechSupport == "Yes"` | so với "No" và "No internet service" |
| `paperless` | `PaperlessBilling == "Yes"` | |

Đúng thứ tự cột như bảng (hằng `FEATURES`), cột 0/1 kiểu `int`, hai cột chia kiểu `float`, index giữ như `raw`.
`y` là Series `int` tên `churned` (1 khi `Churn == "Yes"`).

In [ ]:
FEATURES = ["contract_one_year", "contract_two_year", "tenure_years", "fiber_optic", "no_internet",
            "electronic_check", "monthly_charges_10", "senior", "tech_support", "paperless"]


def build_features(raw: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """X (10 cột FEATURES) và y (int, tên churned)."""
    X = pd.DataFrame({
        "contract_one_year": (raw["Contract"] == "One year").astype(int),
        "contract_two_year": (raw["Contract"] == "Two year").astype(int),
        "tenure_years": raw["tenure"] / 12,
        "fiber_optic": (raw["InternetService"] == "Fiber optic").astype(int),
        "no_internet": (raw["InternetService"] == "No").astype(int),
        "electronic_check": (raw["PaymentMethod"] == "Electronic check").astype(int),
        "monthly_charges_10": raw["MonthlyCharges"] / 10,
        "senior": raw["SeniorCitizen"].astype(int),
        "tech_support": (raw["TechSupport"] == "Yes").astype(int),
        "paperless": (raw["PaperlessBilling"] == "Yes").astype(int),
    }, index=raw.index)[FEATURES]
    y = (raw["Churn"] == "Yes").astype(int).rename("churned")
    return X, y


def split_data(X: pd.DataFrame, y: pd.Series):
    """Chia train/test 75/25, giữ tỉ lệ churn (stratify), cố định seed. Đã viết sẵn."""
    return train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

## 2. Fit và đọc odds ratio

`fit_logit(X, y)`: logistic regression **không regularization** bằng `statsmodels`: `sm.Logit(y, sm.add_constant(X))`,
`.fit(disp=0)`. Trả kết quả fit (có `.params`, `.conf_int()`, `.pvalues`, `.predict`).

`odds_ratio_table(result) -> DataFrame`: index là tên đặc trưng (**bỏ** `const`), cột đúng thứ tự `coef`, `odds_ratio`
(= exp(coef)), `ci_low`, `ci_high` (exp của khoảng tin cậy 95%), `p_value`, `odds_change_pct` (= (odds_ratio − 1) × 100),
sắp theo `odds_ratio` tăng dần.

In [ ]:
def fit_logit(X: pd.DataFrame, y: pd.Series):
    """Logistic regression có hệ số chặn, không regularization (statsmodels)."""
    return sm.Logit(y, sm.add_constant(X, has_constant="add")).fit(disp=0)


def odds_ratio_table(result) -> pd.DataFrame:
    """coef, odds_ratio, ci_low, ci_high, p_value, odds_change_pct; bỏ const; sắp odds_ratio tăng dần."""
    ci = result.conf_int()
    out = pd.DataFrame({
        "coef": result.params,
        "odds_ratio": np.exp(result.params),
        "ci_low": np.exp(ci[0]),
        "ci_high": np.exp(ci[1]),
        "p_value": result.pvalues,
    }).drop(index="const")
    out["odds_change_pct"] = (out["odds_ratio"] - 1) * 100
    return out.sort_values("odds_ratio")

In [ ]:
# explore: fit trên train, đọc bảng odds ratio
X, y = build_features(raw)
X_train, X_test, y_train, y_test = split_data(X, y)
result = fit_logit(X_train, y_train)
pd.set_option("display.width", 160)
print(odds_ratio_table(result).round(3))

## 3. Lift theo decile trên tập test

`decile_table(y_true, score, n_bins=10) -> DataFrame`: sắp khách theo `score` **giảm dần** (sắp ổn định: hoà điểm thì giữ
thứ tự đầu vào), chia thành `n_bins` nhóm theo **vị trí**: khách ở vị trí `i` (0, 1, ...) thuộc nhóm
`i * n_bins // n + 1`. Index tên `decile` (1 = điểm cao nhất), cột đúng thứ tự:

| Cột | Ý nghĩa |
| --- | --- |
| `n` | số khách của nhóm |
| `churners` | số khách rời bỏ |
| `rate` | `churners / n` |
| `lift` | `rate` chia tỉ lệ rời bỏ chung của toàn bộ `y_true` |
| `cum_capture` | tỉ trọng khách rời bỏ nằm trong nhóm này và các nhóm điểm cao hơn |

In [ ]:
def decile_table(y_true, score, n_bins: int = 10) -> pd.DataFrame:
    """n, churners, rate, lift, cum_capture theo nhóm điểm (1 = điểm cao nhất)."""
    yt = np.asarray(y_true).astype(int)
    sc = np.asarray(score, dtype=float)
    order = np.argsort(-sc, kind="stable")
    n = len(yt)
    groups = np.arange(n) * n_bins // n + 1
    df = pd.DataFrame({"decile": groups, "churned": yt[order]})
    out = df.groupby("decile")["churned"].agg(n="size", churners="sum")
    out["rate"] = out["churners"] / out["n"]
    out["lift"] = out["rate"] / yt.mean()
    out["cum_capture"] = out["churners"].cumsum() / yt.sum()
    return out

In [ ]:
# explore: model xếp hạng tốt cỡ nào trên test, và so với quy tắc ở cùng cỡ danh sách
from sklearn.metrics import roc_auc_score

p_test = result.predict(sm.add_constant(X_test, has_constant="add"))
print(f"ROC-AUC test: {roc_auc_score(y_test, p_test):.4f}")
deciles = decile_table(y_test, p_test)
print(deciles.round(3))

rule = (X_test.contract_one_year == 0) & (X_test.contract_two_year == 0) & (X_test.tenure_years <= 1)
k = int(rule.sum())
top_k = np.argsort(-p_test.to_numpy(), kind="stable")[:k]
print(f"quy tắc: gọi {k} khách, bắt {y_test[rule].sum()} khách rời ({y_test[rule].sum() / y_test.sum():.1%})")
print(f"model, cùng {k} khách điểm cao nhất: bắt {y_test.to_numpy()[top_k].sum()} khách rời "
      f"({y_test.to_numpy()[top_k].sum() / y_test.sum():.1%})")

In [ ]:
# explore: vẽ tỉ lệ rời bỏ theo decile, kèm đường tỉ lệ chung
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(deciles.index, deciles["rate"], color="#4c78a8")
ax.axhline(y_test.mean(), color="#e45756", linestyle="--", label=f"tỉ lệ chung {y_test.mean():.1%}")
for d, (r, c) in deciles[["rate", "cum_capture"]].iterrows():
    ax.annotate(f"{r:.0%}\nphủ {c:.0%}", (d, r), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=7)
ax.set_xticks(deciles.index)
ax.set_xlabel("decile điểm rủi ro (1 = cao nhất)")
ax.set_ylabel("tỉ lệ rời bỏ")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
ax.set_title("Model dồn khách sắp rời vào 3 decile đầu", loc="left")
fig.tight_layout()
plt.show()

Viết ra (cho chính bạn, trước khi làm trắc nghiệm):

1. Ba câu cho Head of Retention, mỗi câu một yếu tố, bằng ngôn ngữ odds ("so với…, cùng các yếu tố khác…").
2. Một yếu tố bạn **không** nên diễn giải là nguyên nhân, và vì sao.
3. Ở cùng số cuộc gọi với quy tắc, model bắt thêm bao nhiêu khách rời? Cần thêm thông tin gì để quy ra tiền?

Xong thì mở app, làm phần **trắc nghiệm** của bài.